In [6]:
import pandas as pd
import json
import re
import time

# =============================
# LOAD FILE
# =============================
df = pd.read_csv(r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv", low_memory=False)

results = []

# =============================
# HELPERS
# =============================
def load_json(x):
    try:
        return json.loads(str(x))
    except:
        try:
            import ast
            return ast.literal_eval(str(x))
        except:
            return None

def extract_blocks(data):
    blocks = []
    if isinstance(data, dict):
        if "data" in data:
            blocks.append(data["data"])
        for v in data.values():
            if isinstance(v, dict) and "data" in v:
                blocks.append(v["data"])
    return blocks

def num(x):
    try:
        return float(str(x).replace(",", "").split("/")[0])
    except:
        return 0

def dt(x):
    return pd.to_datetime(x, errors="coerce", dayfirst=True)

# =============================
# MAIN LOOP
# =============================
for raw in df["api_response"]:

    data = load_json(raw)
    if not data:
        continue

    for b in extract_blocks(data):

        pan = b.get("pan")

        if not pan or not re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', pan):
            continue

        try:
            tl = b["credit_report"]["CIR-REPORT-FILE"]["REPORT-DATA"]["STANDARD-DATA"]["TRADELINES"]
        except:
            continue

        if not tl:
            continue

        t = pd.DataFrame(tl)

        # =============================
        # CLEAN
        # =============================
        for c in ["CURRENT-BAL","CREDIT-LIMIT","DISBURSED-AMT","OVERDUE-AMT","WRITE-OFF-AMT","INSTALLMENT-AMT"]:
            t[c] = t.get(c, 0).apply(num)

        for c in ["DISBURSED-DT","CLOSED-DT","REPORTED-DT"]:
            t[c] = dt(t.get(c))

        acct = t["ACCT-TYPE"].astype(str)

        # =============================
        # FLAGS
        # =============================
        is_cc = acct.str.contains("CREDIT CARD", case=False, na=False)
        is_pl = acct.str.contains("PERSONAL", case=False, na=False)
        is_cl = acct.str.contains("CONSUMER", case=False, na=False)
        is_gl = acct.str.contains("GOLD", case=False, na=False)
        is_hl = acct.str.contains("HOME", case=False, na=False)
        is_auto = acct.str.contains("AUTO|CAR|WHEELER", case=False, na=False)
        is_od = acct.str.contains("OD|OVERDRAFT", case=False, na=False)

        active = t["CLOSED-DT"].isna()
        closed = t["CLOSED-DT"].notna()
        overdue = t["OVERDUE-AMT"] > 0

        unsecured = t.get("SECURITY-STATUS","").astype(str).str.contains("UN", case=False, na=False)
        secured = ~unsecured

        non_cc_cd_gl = ~(is_cc | is_cl | is_gl)

        today = pd.Timestamp.today()
        last12 = today - pd.DateOffset(months=12)
        last24 = today - pd.DateOffset(months=24)
        last6 = today - pd.DateOffset(months=6)

        # =============================
        # SORT FOR DATE
        # =============================
        t_open = t.sort_values("DISBURSED-DT")
        t_close = t[t["CLOSED-DT"].notna()].sort_values("CLOSED-DT")

        # =============================
        # RESULT (50 VARIABLES)
        # =============================
        result = {

            "pan": pan,

            # DATE
            "days_since_first_opened": (today - t_open["DISBURSED-DT"].min()).days if t["DISBURSED-DT"].notna().any() else 0,
            "days_since_last_opened": (today - t_open["DISBURSED-DT"].max()).days if t["DISBURSED-DT"].notna().any() else 0,
            "days_since_first_closed": (today - t_close["CLOSED-DT"].min()).days if len(t_close)>0 else 0,
            "days_since_last_closed": (today - t_close["CLOSED-DT"].max()).days if len(t_close)>0 else 0,
            "first_account_type_opened": t_open["ACCT-TYPE"].iloc[0] if len(t_open)>0 else None,
            "last_account_type_opened": t_open["ACCT-TYPE"].iloc[-1] if len(t_open)>0 else None,

            # COUNTS
            "cnt_opened_accounts_last_12mo": (t["DISBURSED-DT"]>=last12).sum(),
            "cnt_closed_accounts_last_12mo": (t["CLOSED-DT"]>=last12).sum(),
            "cnt_all_accounts": len(t),
            "cnt_closed_accounts": closed.sum(),
            "cnt_active_accounts": active.sum(),

            # AMOUNT
            "total_outstanding_amount": t["CURRENT-BAL"].sum(),
            "sum_tradelines_balance": t["DISBURSED-AMT"].sum(),

            # SECURED
            "cnt_secured_loans": secured.sum(),
            "cnt_secured_closed_loans": (secured & closed).sum(),
            "cnt_secured_active_loans": (secured & active).sum(),

            "total_outstanding_amount_secured": t.loc[secured,"CURRENT-BAL"].sum(),
            "total_sanctioned_balance_secured": t.loc[secured,"DISBURSED-AMT"].sum(),

            # UNSECURED
            "cnt_unsecured_loans": unsecured.sum(),
            "cnt_unsecured_closed_loans": (unsecured & closed).sum(),
            "cnt_unsecured_active_loans": (unsecured & active).sum(),

            "total_outstanding_amount_unsecured": t.loc[unsecured,"CURRENT-BAL"].sum(),
            "total_sanctioned_balance_unsecured": t.loc[unsecured,"DISBURSED-AMT"].sum(),

            # CC / OD
            "total_outstanding_amount_cc": t.loc[is_cc,"CURRENT-BAL"].sum(),
            "total_sanctioned_amount_cc": t.loc[is_cc,"CREDIT-LIMIT"].sum(),

            "total_outstanding_amount_od": t.loc[is_od,"CURRENT-BAL"].sum(),
            "total_sanctioned_amount_od": t.loc[is_od,"DISBURSED-AMT"].sum(),

            # ACTIVE SPLIT
            "total_outstanding_amount_active_pl_bl":
                t.loc[active & (is_pl | is_cl),"CURRENT-BAL"].sum(),

            "total_sanctioned_amount_active_pl_bl":
                t.loc[active & (is_pl | is_cl),"DISBURSED-AMT"].sum(),

            "total_outstanding_amount_active_non_cc_cd_gl":
                t.loc[active & non_cc_cd_gl,"CURRENT-BAL"].sum(),

            "total_sanctioned_active_amount_non_cc_cd_gl":
                t.loc[active & non_cc_cd_gl,"DISBURSED-AMT"].sum(),

            # NON CC
            "total_outstanding_amount_non_cc_cd_gl":
                t.loc[non_cc_cd_gl,"CURRENT-BAL"].sum(),

            # WRITE OFF
            "total_writtenoff_amt": t["WRITE-OFF-AMT"].sum(),

            # OBLIGATION
            "total_bureau_obligation": t["INSTALLMENT-AMT"].sum(),

            # UTILIZATION
            "cc_utilization_percentage":
                (t.loc[is_cc,"CURRENT-BAL"].sum() /
                 t.loc[is_cc,"CREDIT-LIMIT"].sum()*100)
                if t.loc[is_cc,"CREDIT-LIMIT"].sum()>0 else 0,

            # OVERDUE
            "cnt_overdue_accounts": overdue.sum(),
            "cnt_active_overdue_accounts": (active & overdue).sum(),

            "cnt_overdue_accounts_last_30days":
                (overdue & (t["REPORTED-DT"] >= today - pd.Timedelta(days=30))).sum(),

            "cnt_overdue_accounts_last_1day":
                (overdue & (t["REPORTED-DT"] >= today - pd.Timedelta(days=1))).sum(),

            # LOAN TYPES
            "cnt_gold_loans": is_gl.sum(),
            "cnt_personal_loans": is_pl.sum(),
            "cnt_consumer_loans": is_cl.sum(),
            "cnt_credit_cards": is_cc.sum(),
            "cnt_car_loans": is_auto.sum(),
            "cnt_home_property_loans": is_hl.sum(),

            "cnt_tw_loans": acct.str.contains("TWO", case=False, na=False).sum(),

            # ACTIVE
            "cnt_active_cc": (is_cc & active).sum(),

            "cnt_active_cc_last_24mo":
                (is_cc & active & (t["DISBURSED-DT"] >= last24)).sum(),
        }

        results.append(result)

# =============================
# SAVE
# =============================
final_df = pd.DataFrame(results)

save_path = rf"C:\Users\Admin\Downloads\FIRST_50_VARIABLES_{int(time.time())}.xlsx"

final_df.to_excel(save_path, index=False)

print("✅ DONE")
print("Saved:", save_path)
print("Shape:", final_df.shape)

✅ DONE
Saved: C:\Users\Admin\Downloads\FIRST_50_VARIABLES_1776167707.xlsx
Shape: (41, 49)


In [7]:
import pandas as pd
import json
import re
import time

# =============================
# LOAD FILE
# =============================
df = pd.read_csv(r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv", low_memory=False)

results = []

# =============================
# HELPERS
# =============================
def load_json(x):
    try:
        return json.loads(str(x))
    except:
        try:
            import ast
            return ast.literal_eval(str(x))
        except:
            return None

def extract_blocks(data):
    blocks = []
    if isinstance(data, dict):
        if "data" in data:
            blocks.append(data["data"])
        for v in data.values():
            if isinstance(v, dict) and "data" in v:
                blocks.append(v["data"])
    return blocks

def num(x):
    try:
        return float(str(x).replace(",", "").split("/")[0])
    except:
        return 0

def dt(x):
    return pd.to_datetime(x, errors="coerce", dayfirst=True)

# =============================
# LOOP
# =============================
for raw in df["api_response"]:

    data = load_json(raw)
    if not data:
        continue

    for b in extract_blocks(data):

        pan = b.get("pan")
        if not pan or not re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', pan):
            continue

        try:
            tl = b["credit_report"]["CIR-REPORT-FILE"]["REPORT-DATA"]["STANDARD-DATA"]["TRADELINES"]
        except:
            continue

        if not tl:
            continue

        t = pd.DataFrame(tl)

        # CLEAN
        for c in ["CURRENT-BAL","CREDIT-LIMIT","DISBURSED-AMT","OVERDUE-AMT","WRITE-OFF-AMT","INSTALLMENT-AMT"]:
            t[c] = t.get(c, 0).apply(num)

        for c in ["DISBURSED-DT","CLOSED-DT","REPORTED-DT"]:
            t[c] = dt(t.get(c))

        acct = t["ACCT-TYPE"].astype(str)

        # FLAGS
        is_cc = acct.str.contains("CREDIT CARD", case=False, na=False)
        is_pl = acct.str.contains("PERSONAL", case=False, na=False)
        is_cl = acct.str.contains("CONSUMER", case=False, na=False)
        is_gl = acct.str.contains("GOLD", case=False, na=False)
        is_hl = acct.str.contains("HOME", case=False, na=False)

        active = t["CLOSED-DT"].isna()
        closed = t["CLOSED-DT"].notna()
        overdue = t["OVERDUE-AMT"] > 0

        unsecured = t.get("SECURITY-STATUS","").astype(str).str.contains("UN", case=False, na=False)
        secured = ~unsecured

        non_cc_cd_gl = ~(is_cc | is_cl | is_gl)

        ownership = t.get("OWNERSHIP-TYPE","").astype(str)
        is_joint = ownership.str.contains("INDIVIDUAL|JOINT", case=False, na=False)

        suit = t.get("SUIT-FILED-WILFUL-DEFAULT-STATUS","").astype(str).str.contains("SUIT", case=False, na=False)
        writtenoff = t["WRITE-OFF-AMT"] > 0
        settled = t.get("WRITTEN-OFF-SETTLED-STATUS","").astype(str).str.contains("SETTLED", case=False, na=False)

        sw = writtenoff | settled

        today = pd.Timestamp.today()
        last12 = today - pd.DateOffset(months=12)
        last24 = today - pd.DateOffset(months=24)
        last6 = today - pd.DateOffset(months=6)

        t["vintage_months"] = ((today - t["DISBURSED-DT"]).dt.days / 30)

        result = {

            "pan": pan,

            # ACTIVE COUNTS
            "cnt_active_hl_joint_individual_ownership_last_24mo":
                (is_hl & active & is_joint & (t["DISBURSED-DT"]>=last24)).sum(),

            "cnt_active_hl": (is_hl & active).sum(),
            "cnt_active_pl_cd_loan": (active & (is_pl | is_cl)).sum(),
            "cnt_active_non_cc_cd_gl": (active & non_cc_cd_gl).sum(),
            "cnt_active_overdue_non_cc_cd_gl": (active & overdue & non_cc_cd_gl).sum(),

            # WRITE OFF / SUIT
            "cnt_writtenoff": writtenoff.sum(),

            "cnt_suit_filed_wilful_default_last_24_mo":
                (suit & (t["REPORTED-DT"]>=last24)).sum(),

            "cnt_settlement_writtenoff_default_last24months":
                (sw & (t["REPORTED-DT"]>=last24)).sum(),

            # NON CC
            "cnt_non_cc_cd_gl_active_accounts": (active & non_cc_cd_gl).sum(),
            "cnt_bl_opened_last6months": (t["DISBURSED-DT"]>=last6).sum(),
            "cnt_non_cc_all_accounts": (~is_cc).sum(),
            "cnt_non_cc_cd_gl_all_accounts": non_cc_cd_gl.sum(),
            "cnt_non_cc_active_accounts": (~is_cc & active).sum(),

            # MIX
            "cnt_individual_joint_non_cc_unsecured_active_loans":
                (is_joint & unsecured & active & ~is_cc).sum(),

            "cnt_overdue_loans": overdue.sum(),
            "cnt_pl_bl_opened": (is_pl | is_cl).sum(),
            "cnt_pl_loan_serviced": is_pl.sum(),

            # AMOUNT FLAGS
            "cnt_unsecured_active_loans_amount_gte10k":
                (unsecured & active & (t["DISBURSED-AMT"]>=10000)).sum(),

            "cnt_active_outstanding_amount_gt_20k":
                (active & (t["CURRENT-BAL"]>20000)).sum(),

            "cnt_active_outstanding_amount_gt_20k_non_cc_cd_gl":
                (active & non_cc_cd_gl & (t["CURRENT-BAL"]>20000)).sum(),

            # HISTORY
            "history_month_cnt":
                ((today - t["DISBURSED-DT"].min()).days/30) if t["DISBURSED-DT"].notna().any() else 0,

            "history_year_cnt":
                ((today - t["DISBURSED-DT"].min()).days/365) if t["DISBURSED-DT"].notna().any() else 0,

            "history_year_cnt_non_gl":
                ((today - t.loc[~is_gl,"DISBURSED-DT"].min()).days/365)
                if t.loc[~is_gl,"DISBURSED-DT"].notna().any() else 0,

            # MAX
            "max_loan_amount_serviced_last2year":
                t.loc[t["DISBURSED-DT"]>=last24,"DISBURSED-AMT"].max(),

            "max_loan_amount_serviced_last6months":
                t.loc[t["DISBURSED-DT"]>=last6,"DISBURSED-AMT"].max(),

            "max_credit_card_limit_last2year":
                t.loc[(is_cc & (t["DISBURSED-DT"]>=last24)),"CREDIT-LIMIT"].max(),

            "max_emi_serviced_last2year":
                t.loc[t["DISBURSED-DT"]>=last24,"INSTALLMENT-AMT"].max(),

            "max_overdue": t["OVERDUE-AMT"].max(),

            "tradeline_max_balance": t["DISBURSED-AMT"].max(),

            "tradeline_max_balance_24months":
                t.loc[t["DISBURSED-DT"]>=last24,"DISBURSED-AMT"].max(),

            # CC ADVANCED
            "max_credit_card_limit_last2year_reported_date":
                t.loc[is_cc].sort_values("CREDIT-LIMIT",ascending=False)["REPORTED-DT"].astype(str).head(1).values[0]
                if is_cc.sum()>0 else None,

            "max_limit_active_cc_vintage_2plus_year":
                t.loc[(is_cc & active & (t["vintage_months"]>=24)),"CREDIT-LIMIT"].max(),

            # HIGH BALANCE
            "max_unsecured_loans_highbalance_non_cc_cd_gl_12mo":
                t.loc[(unsecured & non_cc_cd_gl & (t["DISBURSED-DT"]>=last12)),"DISBURSED-AMT"].max(),

            "max_secured_loans_highbalance_non_cc_cd_gl_12mo":
                t.loc[(secured & non_cc_cd_gl & (t["DISBURSED-DT"]>=last12)),"DISBURSED-AMT"].max(),

            "max_sanctioned_amount_loan_non_gl_6mo":
                t.loc[(~is_gl & (t["DISBURSED-DT"]>=last6)),"DISBURSED-AMT"].max(),

            "max_sanctioned_amount_active_loan_6mo":
                t.loc[(active & (t["DISBURSED-DT"]>=last6)),"DISBURSED-AMT"].max(),

            # SETTLEMENT
            "settlement_writtenoff": sw.sum(),

            "settlement_writtenoff_last12months":
                (sw & (t["REPORTED-DT"]>=last12)).sum(),

            "settlement_writtenoff_last24months":
                (sw & (t["REPORTED-DT"]>=last24)).sum(),

            "settlement_writtenoff_individual_joint_24months":
                (sw & is_joint & (t["REPORTED-DT"]>=last24)).sum(),

            "settlement_writtenoff_non_cc_individual_joint_12months":
                (sw & is_joint & non_cc_cd_gl & (t["REPORTED-DT"]>=last12)).sum(),

            "settlement_writtenoff_non_cc_individual_joint_18months":
                (sw & is_joint & non_cc_cd_gl &
                 (t["REPORTED-DT"]>=today-pd.DateOffset(months=18))).sum(),

            "settlement_writtenoff_non_cc_individual_joint_24months":
                (sw & is_joint & non_cc_cd_gl & (t["REPORTED-DT"]>=last24)).sum(),

            "cnt_settlement_writtenoff_suitfiled_last24months":
                ((sw | suit) & (t["REPORTED-DT"]>=last24)).sum(),

            "is_settled_writtenoff_suitfiled_last24months":
                int(((sw | suit) & (t["REPORTED-DT"]>=last24)).sum()>0),

            "settled_written_off_flag":
                int(sw.sum()>0),

            "is_suitfiled_wilfuldefault_last24months":
                int((suit & (t["REPORTED-DT"]>=last24)).sum()>0),

            # EXTRA
            "total_outstanding_amount_non_hl_al_cc_cd_gl":
                t.loc[~is_hl & ~is_cc & ~is_cl & ~is_gl,"CURRENT-BAL"].sum(),

            "total_sanctioned_amount_non_hl_al_cc_cd_gl":
                t.loc[~is_hl & ~is_cc & ~is_cl & ~is_gl,"DISBURSED-AMT"].sum(),
        }

        results.append(result)

# SAVE
final_df = pd.DataFrame(results)

save_path = rf"C:\Users\Admin\Downloads\SECOND_50_VARIABLES_{int(time.time())}.xlsx"

final_df.to_excel(save_path, index=False)

print("✅ DONE")
print("Saved:", save_path)
print("Shape:", final_df.shape)

✅ DONE
Saved: C:\Users\Admin\Downloads\SECOND_50_VARIABLES_1776168353.xlsx
Shape: (41, 50)


In [8]:
import pandas as pd
import json
import re
import time

df = pd.read_csv(r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv", low_memory=False)

results = []

def load_json(x):
    try:
        return json.loads(str(x))
    except:
        try:
            import ast
            return ast.literal_eval(str(x))
        except:
            return None

def extract_blocks(data):
    blocks = []
    if isinstance(data, dict):
        if "data" in data:
            blocks.append(data["data"])
        for v in data.values():
            if isinstance(v, dict) and "data" in v:
                blocks.append(v["data"])
    return blocks

def num(x):
    try:
        return float(str(x).replace(",", "").split("/")[0])
    except:
        return 0

def dt(x):
    return pd.to_datetime(x, errors="coerce", dayfirst=True)

for raw in df["api_response"]:

    data = load_json(raw)
    if not data:
        continue

    for b in extract_blocks(data):

        pan = b.get("pan")
        if not pan:
            continue

        try:
            tl = b["credit_report"]["CIR-REPORT-FILE"]["REPORT-DATA"]["STANDARD-DATA"]["TRADELINES"]
        except:
            continue

        if not tl:
            continue

        t = pd.DataFrame(tl)

        for c in ["CURRENT-BAL","CREDIT-LIMIT","DISBURSED-AMT","OVERDUE-AMT","WRITE-OFF-AMT","INSTALLMENT-AMT"]:
            t[c] = t.get(c, 0).apply(num)

        for c in ["DISBURSED-DT","CLOSED-DT","REPORTED-DT"]:
            t[c] = dt(t.get(c))

        acct = t["ACCT-TYPE"].astype(str)

        is_cc = acct.str.contains("CREDIT CARD", case=False, na=False)
        is_pl = acct.str.contains("PERSONAL", case=False, na=False)
        is_cl = acct.str.contains("CONSUMER", case=False, na=False)
        is_gl = acct.str.contains("GOLD", case=False, na=False)
        is_hl = acct.str.contains("HOME", case=False, na=False)
        is_auto = acct.str.contains("AUTO|CAR|WHEELER", case=False, na=False)
        is_od = acct.str.contains("OD|OVERDRAFT", case=False, na=False)

        active = t["CLOSED-DT"].isna()
        overdue = t["OVERDUE-AMT"] > 0

        unsecured = t.get("SECURITY-STATUS","").astype(str).str.contains("UN", case=False, na=False)
        non_cc_cd_gl = ~(is_cc | is_cl | is_gl)

        suit = t.get("SUIT-FILED-WILFUL-DEFAULT-STATUS","").astype(str).str.contains("SUIT", case=False, na=False)
        writtenoff = t["WRITE-OFF-AMT"] > 0
        settled = t.get("WRITTEN-OFF-SETTLED-STATUS","").astype(str).str.contains("SETTLED", case=False, na=False)

        sw = writtenoff | settled

        today = pd.Timestamp.today()
        last12 = today - pd.DateOffset(months=12)
        last24 = today - pd.DateOffset(months=24)
        last36 = today - pd.DateOffset(months=36)
        last6 = today - pd.DateOffset(months=6)

        t["vintage"] = ((today - t["DISBURSED-DT"]).dt.days / 30)

        result = {

            "pan": pan,

            # FLAGS
            "is_suitfiled_willfuldefault_writtenoffstatus_last24months":
                int(((suit | writtenoff) & (t["REPORTED-DT"]>=last24)).sum()>0),

            "written_off_gt_1000_flag":
                int((t["WRITE-OFF-AMT"]>1000).sum()>0),

            # AMOUNT
            "amt_pos_all_loans": t["CURRENT-BAL"].sum(),
            "amt_pos_unsecured_loans": t.loc[unsecured,"CURRENT-BAL"].sum(),

            "new_open_acc_last30days":
                (t["DISBURSED-DT"]>=today-pd.Timedelta(days=30)).sum(),

            "amts_writtenoff_last24months":
                t.loc[(writtenoff & (t["REPORTED-DT"]>=last24)),"WRITE-OFF-AMT"].sum(),

            "sum_active_overdue_amount":
                t.loc[(active & overdue),"OVERDUE-AMT"].sum(),

            # NEW ACCOUNTS
            "new_open_acc_gt_1lac_non_cc_last60days":
                (non_cc_cd_gl & (t["DISBURSED-AMT"]>100000) &
                 (t["DISBURSED-DT"]>=today-pd.Timedelta(days=60))).sum(),

            "new_open_acc_lte_1lac_non_cc_last60days":
                (non_cc_cd_gl & (t["DISBURSED-AMT"]<=100000) &
                 (t["DISBURSED-DT"]>=today-pd.Timedelta(days=60))).sum(),

            "new_open_acc_non_cc_last60days":
                (non_cc_cd_gl &
                 (t["DISBURSED-DT"]>=today-pd.Timedelta(days=60))).sum(),

            # ACTIVE SANCTION
            "sanction_active_pl_cd_loan":
                t.loc[(active & (is_pl | is_cl)),"DISBURSED-AMT"].sum(),

            "prin_run_off": t["WRITE-OFF-AMT"].sum(),

            "od_utilization_perc":
                (t.loc[is_od,"CURRENT-BAL"].sum() /
                 t.loc[is_od,"DISBURSED-AMT"].sum()*100)
                if t.loc[is_od,"DISBURSED-AMT"].sum()>0 else 0,

            "sum_active_overdue_amount_non_cc_cd_gl":
                t.loc[(active & overdue & non_cc_cd_gl),"OVERDUE-AMT"].sum(),

            "sum_outstanding_bal_active_cc":
                t.loc[(active & is_cc),"CURRENT-BAL"].sum(),

            # MAX ACTIVE VINTAGE
            "max_active_hl_amount_vintage_12months":
                t.loc[(is_hl & active & (t["DISBURSED-DT"]>=last12)),"DISBURSED-AMT"].max(),

            "max_active_auto_loan_amount_vintage_12months":
                t.loc[(is_auto & active & (t["DISBURSED-DT"]>=last12)),"DISBURSED-AMT"].max(),

            "max_active_cc_amount_vintage_12months":
                t.loc[(is_cc & active & (t["DISBURSED-DT"]>=last12)),"CREDIT-LIMIT"].max(),

            "max_active_unsecured_amount_vintage_12months":
                t.loc[(unsecured & active & (t["DISBURSED-DT"]>=last12)),"DISBURSED-AMT"].max(),

            # SUM
            "sum_tradelines_current_balance": t["CURRENT-BAL"].sum(),
            "sum_auto_loan_balance": t.loc[is_auto,"CURRENT-BAL"].sum(),

            # COUNT
            "cnt_hl_vintage_12month":
                (is_hl & (t["DISBURSED-DT"]>=last12)).sum(),

            "cnt_cc_vintage_24month":
                (is_cc & (t["DISBURSED-DT"]>=last24)).sum(),

            "cnt_cd_loan_lt_25k":
                (is_cl & (t["DISBURSED-AMT"]<25000)).sum(),

            "settlement_writtenoff_last36months":
                (sw & (t["REPORTED-DT"]>=last36)).sum(),

            "max_unsecured_loans_highbalance":
                t.loc[unsecured,"DISBURSED-AMT"].max(),

            "cnt_loans_non_cc_cl":
                (~is_cc).sum(),

            "cnt_unsecured_loan_opened_last_90days":
                (unsecured & (t["DISBURSED-DT"]>=today-pd.Timedelta(days=90))).sum(),

            "cnt_unsecured_loan_non_cc_cl_opened_last_30days":
                (unsecured & non_cc_cd_gl &
                 (t["DISBURSED-DT"]>=today-pd.Timedelta(days=30))).sum(),

            "cnt_unsecured_loan_non_cc_cl_opened_last_60days":
                (unsecured & non_cc_cd_gl &
                 (t["DISBURSED-DT"]>=today-pd.Timedelta(days=60))).sum(),

            "cnt_non_cc_cd_gl_active_overdue_accounts":
                (active & overdue & non_cc_cd_gl).sum(),

            "cnt_loan_pos_20k_plus_non_cc_cd_gl":
                (non_cc_cd_gl & (t["CURRENT-BAL"]>20000)).sum(),

            "sum_high_balance_last_6month":
                t.loc[t["DISBURSED-DT"]>=last6,"DISBURSED-AMT"].sum(),

            "sum_tradelines_balance_non_cc_cd_gl":
                t.loc[non_cc_cd_gl,"DISBURSED-AMT"].sum(),

            "sum_tradelines_current_balance_non_cc_cd_gl":
                t.loc[non_cc_cd_gl,"CURRENT-BAL"].sum(),

            "sum_active_tradelines_current_balance_non_cc_cd_gl":
                t.loc[(active & non_cc_cd_gl),"CURRENT-BAL"].sum(),

            "min_amount_unsecured_last_6months":
                t.loc[(unsecured & (t["DISBURSED-DT"]>=last6)),"DISBURSED-AMT"].min(),

            "min_amount_unsecured_non_cc_last_6months":
                t.loc[(unsecured & non_cc_cd_gl &
                       (t["DISBURSED-DT"]>=last6)),"DISBURSED-AMT"].min(),
        }

        results.append(result)

# SAVE
final_df = pd.DataFrame(results)

save_path = rf"C:\Users\Admin\Downloads\FINAL_47_VARIABLES_{int(time.time())}.xlsx"
final_df.to_excel(save_path, index=False)

print("✅ DONE")
print("Saved:", save_path)
print("Shape:", final_df.shape)

✅ DONE
Saved: C:\Users\Admin\Downloads\FINAL_47_VARIABLES_1776168852.xlsx
Shape: (41, 39)


In [10]:
import pandas as pd
import json
import re
import time

df = pd.read_csv(r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv", low_memory=False)

results = []

def load_json(x):
    try:
        return json.loads(str(x))
    except:
        import ast
        try:
            return ast.literal_eval(str(x))
        except:
            return None

def extract_blocks(data):
    blocks = []
    if isinstance(data, dict):
        if "data" in data:
            blocks.append(data["data"])
        for v in data.values():
            if isinstance(v, dict) and "data" in v:
                blocks.append(v["data"])
    return blocks

def num(x):
    try:
        return float(str(x).replace(",", "").split("/")[0])
    except:
        return 0

def dt(x):
    return pd.to_datetime(x, errors="coerce", dayfirst=True)

for raw in df["api_response"]:

    data = load_json(raw)
    if not data:
        continue

    for b in extract_blocks(data):

        pan = b.get("pan")
        if not pan or not re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', pan):
            continue

        try:
            tl = b["credit_report"]["CIR-REPORT-FILE"]["REPORT-DATA"]["STANDARD-DATA"]["TRADELINES"]
        except:
            continue

        if not tl:
            continue

        t = pd.DataFrame(tl)

        for c in ["CURRENT-BAL","CREDIT-LIMIT","DISBURSED-AMT","OVERDUE-AMT","WRITE-OFF-AMT"]:
            t[c] = t.get(c, 0).apply(num)

        for c in ["DISBURSED-DT","REPORTED-DT"]:
            t[c] = dt(t.get(c))

        acct = t["ACCT-TYPE"].astype(str)

        is_cc = acct.str.contains("CREDIT CARD", case=False, na=False)
        is_cl = acct.str.contains("CONSUMER", case=False, na=False)
        is_gl = acct.str.contains("GOLD", case=False, na=False)

        non_cc_cd_gl = ~(is_cc | is_cl | is_gl)

        unsecured = t.get("SECURITY-STATUS","").astype(str).str.contains("UN", case=False, na=False)

        today = pd.Timestamp.today()
        last24 = today - pd.DateOffset(months=24)

        result = {

            "pan": pan,

            # =============================
            # FINAL 9 VARIABLES
            # =============================

            # POS AMOUNT
            "amt_pos_all_loans": t["CURRENT-BAL"].sum(),

            "amt_pos_unsecured_loans":
                t.loc[unsecured, "CURRENT-BAL"].sum(),

            # NEW ACCOUNTS
            "new_open_acc_last30days":
                (t["DISBURSED-DT"] >= today - pd.Timedelta(days=30)).sum(),

            "new_open_acc_non_cc_last60days":
                (non_cc_cd_gl & (t["DISBURSED-DT"] >= today - pd.Timedelta(days=60))).sum(),

            "new_open_acc_gt_1lac_non_cc_last60days":
                (non_cc_cd_gl & (t["DISBURSED-AMT"] > 100000) &
                 (t["DISBURSED-DT"] >= today - pd.Timedelta(days=60))).sum(),

            "new_open_acc_lte_1lac_non_cc_last60days":
                (non_cc_cd_gl & (t["DISBURSED-AMT"] <= 100000) &
                 (t["DISBURSED-DT"] >= today - pd.Timedelta(days=60))).sum(),

            # WRITE OFF
            "amts_writtenoff_last24months":
                t.loc[t["REPORTED-DT"] >= last24, "WRITE-OFF-AMT"].sum(),

            # RUN OFF
            "prin_run_off":
                t["DISBURSED-AMT"].sum() - t["CURRENT-BAL"].sum(),

            # OD UTILIZATION
            "od_utilization_perc":
                (t.loc[is_cc, "CURRENT-BAL"].sum() /
                 t.loc[is_cc, "CREDIT-LIMIT"].sum() * 100)
                if t.loc[is_cc, "CREDIT-LIMIT"].sum() > 0 else 0
        }

        results.append(result)

final_df = pd.DataFrame(results)

save_path = rf"C:\Users\Admin\Downloads\LAST_9_VARIABLES_{int(time.time())}.xlsx"

final_df.to_excel(save_path, index=False)

print("✅ DONE")
print("Saved:", save_path)
print("Shape:", final_df.shape)

✅ DONE
Saved: C:\Users\Admin\Downloads\LAST_9_VARIABLES_1776170132.xlsx
Shape: (41, 10)


In [2]:
import pandas as pd
import json
import ast

# =============================
# LOAD
# =============================
path = r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv"
df = pd.read_csv(path, low_memory=False)

results = []

# =============================
# HELPERS
# =============================
def load_json(x):
    try:
        return json.loads(str(x))
    except:
        try:
            return ast.literal_eval(str(x))
        except:
            return None

def dt(x):
    return pd.to_datetime(x, errors="coerce", dayfirst=True)

# =============================
# LOOP
# =============================
for raw in df["api_response"]:

    data = load_json(raw)
    if not isinstance(data, dict):
        continue

    try:
        block = data["data"]
        pan = block.get("pan")

        enquiries = (
            block.get("credit_report", {})
                 .get("CIR-REPORT-FILE", {})
                 .get("REPORT-DATA", {})
                 .get("STANDARD-DATA", {})
                 .get("INQUIRY-HISTORY", [])
        )
    except:
        continue

    if not enquiries:
        continue

    e = pd.DataFrame(enquiries)

    if e.empty:
        continue

    # =============================
    # CLEAN
    # =============================
    e["INQUIRY-DT"] = dt(e.get("INQUIRY-DT"))
    loan_type = e.get("LOAN-TYPE", "").astype(str)

    today = pd.Timestamp.today()

    last7 = today - pd.Timedelta(days=7)
    last30 = today - pd.Timedelta(days=30)
    last90 = today - pd.Timedelta(days=90)
    last180 = today - pd.Timedelta(days=180)
    last6m = today - pd.DateOffset(months=6)
    last3m = today - pd.DateOffset(months=3)

    # =============================
    # FLAGS
    # =============================
    is_pl = loan_type.str.contains("PERSONAL", case=False, na=False)

    is_cc = loan_type.str.contains("CREDIT CARD", case=False, na=False)

    is_consumer = loan_type.str.contains("CONSUMER", case=False, na=False)

    is_unsecured = is_pl | is_cc | is_consumer

    is_secured = ~is_unsecured

    non_cc = ~is_cc

    # =============================
    # CALCULATIONS
    # =============================
    result = {

        "pan": pan,

        "enquiry_last7days": (e["INQUIRY-DT"] >= last7).sum(),

        "enquiry_last30days": (e["INQUIRY-DT"] >= last30).sum(),

        "enquiry_last90days": (e["INQUIRY-DT"] >= last90).sum(),

        "enquiry_last180days": (e["INQUIRY-DT"] >= last180).sum(),

        "enquiry_last6months": (e["INQUIRY-DT"] >= last6m).sum(),

        "enquiry_pl_last3months":
            (is_pl & (e["INQUIRY-DT"] >= last3m)).sum(),

        "enquiry_unsecured_last6months":
            (is_unsecured & (e["INQUIRY-DT"] >= last6m)).sum(),

        "enquiry_unsecured_last90days":
            (is_unsecured & (e["INQUIRY-DT"] >= last90)).sum(),

        "enquiry_secured_last30days":
            (is_secured & (e["INQUIRY-DT"] >= last30)).sum(),

        "enquiry_secured_last90days":
            (is_secured & (e["INQUIRY-DT"] >= last90)).sum(),

        "enquiry_non_cc_unsecured_last3months":
            (non_cc & is_unsecured & (e["INQUIRY-DT"] >= last3m)).sum(),

        "enquiry_non_cc_unsecured_last90days":
            (non_cc & is_unsecured & (e["INQUIRY-DT"] >= last90)).sum(),

        "cnt_bl_inquries_last_1m":
            (is_unsecured & (e["INQUIRY-DT"] >= last30)).sum(),

        "cnt_enquiry_types_5_9_45_51_61_in_last1month": 0,

        "cnt_enquiry_types_5_9_45_51_61_in_last3months": 0,

        "cnt_enquiry_types_5_9_45_51_61_in_last6months": 0,

        "enquiry_unsecured_last30days":
            (is_unsecured & (e["INQUIRY-DT"] >= last30)).sum(),

        "enquiry_unsecured_last3months":
            (is_unsecured & (e["INQUIRY-DT"] >= last3m)).sum(),

        "cnt_bl_inquiries_last_1m":
            (is_unsecured & (e["INQUIRY-DT"] >= last30)).sum()
    }

    results.append(result)

# =============================
# OUTPUT
# =============================
final_df = pd.DataFrame(results)

print("✅ DONE")
print("PAN count:", final_df.shape[0])
print(final_df.head())

✅ DONE
PAN count: 95
          pan  enquiry_last7days  enquiry_last30days  enquiry_last90days  \
0  KYEPS8060N                  0                   0                   1   
1  FEWPA9169H                  0                   0                   1   
2  ATOPB8483D                  0                   0                   6   
3  BTZPG5521K                  0                   0                   2   
4  GHRPA4842N                  0                   0                   1   

   enquiry_last180days  enquiry_last6months  enquiry_pl_last3months  \
0                    1                    1                       0   
1                    2                    2                       0   
2                   15                   15                       1   
3                   10                   10                       1   
4                    1                    1                       1   

   enquiry_unsecured_last6months  enquiry_unsecured_last90days  \
0                            

In [4]:
# =============================
# FINAL DATAFRAME
# =============================
final_df = pd.DataFrame(results)

# =============================
# KEEP ONLY REQUIRED 19 VARIABLES
# =============================
cols = [
    "pan",
    "enquiry_last7days",
    "enquiry_last30days",
    "enquiry_last90days",
    "enquiry_last180days",
    "enquiry_last6months",
    "enquiry_pl_last3months",
    "enquiry_unsecured_last6months",
    "enquiry_unsecured_last90days",
    "enquiry_secured_last30days",
    "enquiry_secured_last90days",
    "enquiry_non_cc_unsecured_last3months",
    "enquiry_non_cc_unsecured_last90days",
    "cnt_bl_inquries_last_1m",
    "cnt_enquiry_types_5_9_45_51_61_in_last1month",
    "cnt_enquiry_types_5_9_45_51_61_in_last3months",
    "cnt_enquiry_types_5_9_45_51_61_in_last6months",
    "enquiry_unsecured_last30days",
    "enquiry_unsecured_last3months",
    "cnt_bl_inquiries_last_1m"
]

final_df = final_df[cols]

# =============================
# SAVE FILE
# =============================
output_path = r"C:\Users\Admin\Downloads\ENQUIRY_19_VARIABLES.xlsx"

final_df.to_excel(output_path, index=False)

print("✅ DONE")
print("Saved at:", output_path)
print("Shape:", final_df.shape)

✅ DONE
Saved at: C:\Users\Admin\Downloads\ENQUIRY_19_VARIABLES.xlsx
Shape: (95, 20)


In [6]:
import pandas as pd
import json
import ast

# load file
path = r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv"
df = pd.read_csv(path, low_memory=False)

# pick one PAN
pan_to_check = "SMCPS8940F"   # change if needed

# safe json loader
def safe_json(x):
    try:
        return json.loads(x)
    except:
        try:
            return json.loads(x.replace('""','"'))
        except:
            try:
                return ast.literal_eval(x)
            except:
                return None

# get row
row = df[df["pan"] == pan_to_check]["api_response"].iloc[0]

data = safe_json(str(row))

# extract tradelines
block = data.get("data", {})

tradelines = (
    block.get("credit_report", {})
         .get("CIR-REPORT-FILE", {})
         .get("REPORT-DATA", {})
         .get("STANDARD-DATA", {})
         .get("TRADELINES", [])
)

# create dataframe
t = pd.DataFrame(tradelines)

print("✅ Tradelines extracted")
print("Shape:", t.shape)

✅ Tradelines extracted
Shape: (12, 41)


In [7]:
print(t.columns)

Index(['HISTORY', 'ACCT-TYPE', 'CLOSED-DT', 'CASH-LIMIT', 'OBLIGATION',
       'OCCUPATION', 'ACCT-NUMBER', 'CURRENT-BAL', 'OVERDUE-AMT',
       'REPORTED-DT', 'CREDIT-LIMIT', 'DISBURSED-DT', 'WRITE-OFF-DT',
       'DISBURSED-AMT', 'INCOME-AMOUNT', 'INTEREST-RATE', 'ORIGINAL-TERM',
       'SUIT-FILED-DT', 'WRITE-OFF-AMT', 'ACCOUNT-STATUS', 'ACTUAL-PAYMENT',
       'CREDIT-GRANTOR', 'OWNERSHIP-TYPE', 'SETTLEMENT-AMT', 'ACCOUNT-REMARKS',
       'ACCT-IN-DISPUTE', 'INSTALLMENT-AMT', 'LAST-PAYMENT-DT',
       'LINKED-ACCOUNTS', 'SECURITY-STATUS', 'INCOME-FREQUENCY',
       'LAST-PAID-AMOUNT', 'REPAYMENT-TENURE', 'SECURITY-DETAILS',
       'TERM-TO-MATURITY', 'CREDIT-GRANTOR-TYPE', 'CREDIT-GRANTOR-GROUP',
       'INSTALLMENT-FREQUENCY', 'PRINCIPAL-WRITE-OFF-AMT',
       'WRITTEN-OFF-SETTLED-STATUS', 'SUIT-FILED-WILFUL-DEFAULT-STATUS'],
      dtype='object')


In [8]:
print(t["HISTORY"].iloc[0])

[{'NAME': 'COMBINED-PAYMENT-HISTORY', 'DATES': 'Feb:2026|', 'VALUES': '000/STD|'}]


In [9]:
def parse_history(hist):
    dpd_list = []

    if not hist:
        return dpd_list

    # hist is list of dicts
    for block in hist:
        values = block.get("VALUES", "")

        # split months
        parts = values.split("|")

        for p in parts:
            if "/" in p:
                dpd = p.split("/")[0].strip()

                if dpd in ["STD", "XXX", "", "NAN"]:
                    dpd_list.append(0)
                else:
                    try:
                        dpd_list.append(int(dpd))
                    except:
                        dpd_list.append(0)

    return dpd_list

In [10]:
t["DPD_LIST"] = t["HISTORY"].apply(parse_history)

In [11]:
print(t[["HISTORY","DPD_LIST"]].head())

                                             HISTORY                  DPD_LIST
0  [{'NAME': 'COMBINED-PAYMENT-HISTORY', 'DATES':...                       [0]
1  [{'NAME': 'COMBINED-PAYMENT-HISTORY', 'DATES':...              [0, 0, 0, 0]
2  [{'NAME': 'COMBINED-PAYMENT-HISTORY', 'DATES':...             [0, 30, 0, 0]
3  [{'NAME': 'COMBINED-PAYMENT-HISTORY', 'DATES':...  [26, 11, 11, 0, 0, 0, 0]
4  [{'NAME': 'COMBINED-PAYMENT-HISTORY', 'DATES':...   [11, 0, 27, 0, 0, 0, 0]


In [12]:
all_dpd = [x for sub in t["DPD_LIST"] for x in sub]

In [13]:
print("All DPD:", all_dpd)
print("Max DPD:", max(all_dpd) if all_dpd else 0)

All DPD: [0, 0, 0, 0, 0, 0, 30, 0, 0, 26, 11, 11, 0, 0, 0, 0, 11, 0, 27, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 30, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 900, 900, 900, 900, 900, 900, 900, 900, 900, 900, 900, 900, 900, 879, 848, 817, 787, 756, 726, 695, 664, 634, 603, 573, 542, 513, 482, 451, 421, 390, 360, 329, 298, 268, 237, 207, 0, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Max DPD: 900


In [14]:
def parse_history(hist):
    dpd_list = []

    if not hist:
        return dpd_list

    for block in hist:
        values = block.get("VALUES", "")
        parts = values.split("|")

        for p in parts:
            if "/" in p:
                dpd = p.split("/")[0].strip()

                if dpd in ["STD", "XXX", "", "NAN"]:
                    dpd_list.append(0)
                else:
                    try:
                        val = int(dpd)

                        # 🚨 FILTER BAD VALUES
                        if val > 365:
                            val = 0

                        dpd_list.append(val)

                    except:
                        dpd_list.append(0)

    return dpd_list

In [15]:
t["DPD_LIST"] = t["HISTORY"].apply(parse_history)

all_dpd = [x for sub in t["DPD_LIST"] for x in sub]

print("Max DPD:", max(all_dpd))

Max DPD: 360


In [16]:
result = {}

result["max_dpd_ever"] = max(all_dpd) if all_dpd else 0

result["most_recent_dpd"] = all_dpd[0] if all_dpd else 0

result["max_dpd_last_6_mo"] = max(all_dpd[:6]) if all_dpd else 0

result["max_dpd_inlast12months"] = max(all_dpd[:12]) if all_dpd else 0

In [17]:
print(sorted(set(all_dpd))[-10:])

[13, 26, 27, 30, 207, 237, 268, 298, 329, 360]


In [18]:
last3  = all_dpd[:3]
last6  = all_dpd[:6]
last12 = all_dpd[:12]
last24 = all_dpd[:24]
last36 = all_dpd[:36]

In [19]:
result = {}

# basic
result["max_dpd_ever"] = max(all_dpd) if all_dpd else 0
result["most_recent_dpd"] = all_dpd[0] if all_dpd else 0

# time-based
result["max_dpd_last_6_mo"] = max(last6) if last6 else 0
result["max_dpd_inlast12months"] = max(last12) if last12 else 0
result["max_dpd_inlast24months"] = max(last24) if last24 else 0
result["max_dpd_inlast36months"] = max(last36) if last36 else 0

# counts
result["cnt_30plus_dpd_last6months"] = sum(x >= 30 for x in last6)
result["cnt_30plus_dpd_last24months"] = sum(x >= 30 for x in last24)
result["cnt_90plus_dpd_last24months"] = sum(x >= 90 for x in last24)
result["cnt_90plus_dpd_last36months"] = sum(x >= 90 for x in last36)

# zero dpd
result["cnt_0dpd_last_6mo"] = sum(x == 0 for x in last6)
result["cnt_0dpd_last_12mo"] = sum(x == 0 for x in last12)

In [20]:
active = t["CLOSED-DT"].isna()

active_dpd = [
    x for i,row in t.iterrows()
    if active[i]
    for x in row["DPD_LIST"]
]

result["max_dpd_current_active_account"] = max(active_dpd) if active_dpd else 0

In [21]:
acct = t["ACCT-TYPE"].astype(str)

is_cc = acct.str.contains("CARD", case=False, na=False)
is_non_cc = ~is_cc

cc_dpd = [
    x for i,row in t.iterrows() if is_cc[i]
    for x in row["DPD_LIST"]
]

non_cc_dpd = [
    x for i,row in t.iterrows() if is_non_cc[i]
    for x in row["DPD_LIST"]
]

result["max_dpd_cc_inlast6months"] = max(cc_dpd[:6]) if cc_dpd else 0
result["max_dpd_non_cc_inlast6months"] = max(non_cc_dpd[:6]) if non_cc_dpd else 0

In [22]:
sec = t["SECURITY-STATUS"].astype(str)

is_sec = sec.str.contains("SEC", case=False, na=False)
is_unsec = sec.str.contains("UN", case=False, na=False)

secured_dpd = [
    x for i,row in t.iterrows() if is_sec[i]
    for x in row["DPD_LIST"]
]

unsecured_dpd = [
    x for i,row in t.iterrows() if is_unsec[i]
    for x in row["DPD_LIST"]
]

result["max_dpd_secured_ever"] = max(secured_dpd) if secured_dpd else 0
result["max_dpd_unsecured_ever"] = max(unsecured_dpd) if unsecured_dpd else 0

In [23]:
import pandas as pd
import json, ast, re

# =============================
# LOAD
# =============================
path = r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv"
df = pd.read_csv(path, low_memory=False)

# =============================
# HELPERS
# =============================
def safe_json(x):
    try:
        return json.loads(x)
    except:
        try:
            return json.loads(x.replace('""','"'))
        except:
            try:
                return ast.literal_eval(x)
            except:
                return None

def parse_history(hist):
    out = []
    if not hist:
        return out
    for block in hist:
        vals = str(block.get("VALUES",""))
        for p in vals.split("|"):
            if "/" in p:
                d = p.split("/")[0].strip()
                if d in ["STD","XXX","","NAN"]:
                    out.append(0)
                else:
                    try:
                        v = int(d)
                        # cap garbage
                        if v > 365: v = 0
                        out.append(v)
                    except:
                        out.append(0)
    return out

def flatten(list_of_lists):
    return [x for sub in list_of_lists for x in sub]

def max_or_0(x): return max(x) if x else 0
def cnt(x, cond): return sum(1 for v in x if cond(v))

# =============================
# MAIN LOOP
# =============================
rows = []

for raw in df["api_response"]:
    data = safe_json(str(raw))
    if not isinstance(data, dict): 
        continue

    try:
        block = data["data"]
        pan = block.get("pan")

        tradelines = (
            block.get("credit_report", {})
                 .get("CIR-REPORT-FILE", {})
                 .get("REPORT-DATA", {})
                 .get("STANDARD-DATA", {})
                 .get("TRADELINES", [])
        )
    except:
        continue

    if not tradelines:
        continue

    t = pd.DataFrame(tradelines)
    if t.empty:
        continue

    # =============================
    # DPD LIST PER TL
    # =============================
    t["DPD_LIST"] = t["HISTORY"].apply(parse_history)

    # GLOBAL (all TLs flattened)
    all_dpd = flatten(t["DPD_LIST"])

    # windows (CRIF latest first)
    last3  = all_dpd[:3]
    last6  = all_dpd[:6]
    last12 = all_dpd[:12]
    last24 = all_dpd[:24]
    last36 = all_dpd[:36]

    # =============================
    # FLAGS
    # =============================
    acct = t.get("ACCT-TYPE","").astype(str)
    sec  = t.get("SECURITY-STATUS","").astype(str)

    is_cc  = acct.str.contains("CARD", case=False, na=False)
    is_non_cc = ~is_cc

    is_sec = sec.str.contains("SEC", case=False, na=False)
    is_unsec = sec.str.contains("UN", case=False, na=False)

    active = t.get("CLOSED-DT").isna() if "CLOSED-DT" in t.columns else pd.Series([True]*len(t))

    # helper flatten by mask
    def dpd_by(mask):
        return [x for i,row in t.iterrows() if mask[i] for x in row["DPD_LIST"]]

    cc_dpd = dpd_by(is_cc)
    non_cc_dpd = dpd_by(is_non_cc)
    sec_dpd = dpd_by(is_sec)
    unsec_dpd = dpd_by(is_unsec)
    active_dpd = dpd_by(active)
    active_cc_dpd = dpd_by(active & is_cc)
    active_non_cc_dpd = dpd_by(active & is_non_cc)

    # =============================
    # BUILD VARIABLES
    # =============================
    r = {"pan": pan}

    # ---- BASIC
    r["max_dpd_ever"] = max_or_0(all_dpd)
    r["most_recent_dpd"] = all_dpd[0] if all_dpd else 0

    # ---- TIME WINDOWS
    r["max_dpd_last_6_mo"] = max_or_0(last6)
    r["max_dpd_inlast12months"] = max_or_0(last12)
    r["max_dpd_inlast24months"] = max_or_0(last24)
    r["max_dpd_inlast36months"] = max_or_0(last36)

    # ---- ACTIVE
    r["max_dpd_current_active_account"] = max_or_0(active_dpd)
    r["max_dpd_active_account_last_3_mo"] = max_or_0(active_dpd[:3])
    r["max_dpd_active_account_last_6_mo"] = max_or_0(active_dpd[:6])
    r["max_dpd_active_account_last_12_mo"] = max_or_0(active_dpd[:12])
    r["max_dpd_active_account_last_24_mo"] = max_or_0(active_dpd[:24])
    r["max_dpd_active_account_last_36_mo"] = max_or_0(active_dpd[:36])

    # ---- CC / NON-CC
    r["max_dpd_cc_last_3mo"] = max_or_0(cc_dpd[:3])
    r["max_dpd_cc_last_6mo"] = max_or_0(cc_dpd[:6])
    r["max_dpd_cc_last_12mo"] = max_or_0(cc_dpd[:12])

    r["max_dpd_non_cc_last_3mo"] = max_or_0(non_cc_dpd[:3])
    r["max_dpd_non_cc_last_6mo"] = max_or_0(non_cc_dpd[:6])
    r["max_dpd_non_cc_last_12mo"] = max_or_0(non_cc_dpd[:12])

    r["max_dpd_cc_inlast6months"] = max_or_0(cc_dpd[:6])
    r["max_dpd_non_cc_inlast6months"] = max_or_0(non_cc_dpd[:6])

    # ---- SECURED / UNSECURED
    r["max_dpd_secured_ever"] = max_or_0(sec_dpd)
    r["max_dpd_unsecured_ever"] = max_or_0(unsec_dpd)

    r["max_dpd_secured_inlast6months"] = max_or_0(sec_dpd[:6])
    r["max_dpd_unsecured_inlast6months"] = max_or_0(unsec_dpd[:6])

    # ---- COUNTS
    r["cnt_30plus_dpd_last6months"] = cnt(last6, lambda x: x >= 30)
    r["cnt_30plus_dpd_last24months"] = cnt(last24, lambda x: x >= 30)
    r["cnt_90plus_dpd_last24months"] = cnt(last24, lambda x: x >= 90)
    r["cnt_90plus_dpd_last36months"] = cnt(last36, lambda x: x >= 90)

    r["cnt_0dpd_last_6mo"] = cnt(last6, lambda x: x == 0)
    r["cnt_0dpd_last_12mo"] = cnt(last12, lambda x: x == 0)

    # ---- ACTIVE SPLIT
    r["max_dpd_active_cc_last_3mo"] = max_or_0(active_cc_dpd[:3])
    r["max_dpd_active_cc_last_6mo"] = max_or_0(active_cc_dpd[:6])
    r["max_dpd_active_cc_last_12mo"] = max_or_0(active_cc_dpd[:12])

    r["max_dpd_active_non_cc_last_3mo"] = max_or_0(active_non_cc_dpd[:3])
    r["max_dpd_active_non_cc_last_6mo"] = max_or_0(active_non_cc_dpd[:6])
    r["max_dpd_active_non_cc_last_12mo"] = max_or_0(active_non_cc_dpd[:12])

    # ---- INSTALLMENT STATS
    total_inst = len(all_dpd)
    dpd_inst = cnt(all_dpd, lambda x: x > 0)

    r["cnt_dpd_installments"] = dpd_inst
    r["cnt_paid_dpd_installments"] = cnt(all_dpd, lambda x: x > 0)
    r["cnt_unpaid_dpd_installments"] = cnt(all_dpd, lambda x: x > 0)

    r["percentage_installments_defaulted"] = (dpd_inst / total_inst * 100) if total_inst else 0

    # ---- SIMPLE FLAGS
    r["written_off_flag"] = int(
        (t.get("WRITE-OFF-AMT", 0).fillna(0).astype(float).sum() > 0)
    )

    rows.append(r)

# =============================
# OUTPUT
# =============================
final_df = pd.DataFrame(rows)

print("✅ DONE")
print("PAN count:", final_df.shape[0])
print("Columns:", final_df.shape[1])
print(final_df.head())

ValueError: could not convert string to float: '25,117'

In [24]:
import pandas as pd
import json, ast

# =============================
# LOAD
# =============================
path = r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv"
df = pd.read_csv(path, low_memory=False)

# =============================
# HELPERS
# =============================
def safe_json(x):
    try:
        return json.loads(x)
    except:
        try:
            return json.loads(x.replace('""','"'))
        except:
            try:
                return ast.literal_eval(x)
            except:
                return None

def num(x):
    try:
        return float(str(x).replace(",", "").strip())
    except:
        return 0.0

def parse_history(hist):
    dpd_list = []
    if not hist:
        return dpd_list

    for block in hist:
        values = str(block.get("VALUES", ""))

        for p in values.split("|"):
            if "/" in p:
                d = p.split("/")[0].strip()

                if d in ["STD","XXX","","NAN"]:
                    dpd_list.append(0)
                else:
                    try:
                        v = int(d)
                        if v > 365:  # remove garbage
                            v = 0
                        dpd_list.append(v)
                    except:
                        dpd_list.append(0)

    return dpd_list

def flatten(lst):
    return [x for sub in lst for x in sub]

def max0(x): return max(x) if x else 0

# =============================
# MAIN LOOP
# =============================
final = []

for raw in df["api_response"]:

    data = safe_json(str(raw))
    if not isinstance(data, dict):
        continue

    try:
        block = data["data"]
        pan = block.get("pan")

        tradelines = (
            block.get("credit_report", {})
                 .get("CIR-REPORT-FILE", {})
                 .get("REPORT-DATA", {})
                 .get("STANDARD-DATA", {})
                 .get("TRADELINES", [])
        )
    except:
        continue

    if not tradelines:
        continue

    t = pd.DataFrame(tradelines)
    if t.empty:
        continue

    # =============================
    # CLEAN NUMERIC
    # =============================
    num_cols = [
        "CURRENT-BAL","OVERDUE-AMT","CREDIT-LIMIT",
        "DISBURSED-AMT","WRITE-OFF-AMT","INSTALLMENT-AMT"
    ]

    for col in num_cols:
        if col in t.columns:
            t[col] = t[col].apply(num)
        else:
            t[col] = 0

    # =============================
    # DPD EXTRACTION
    # =============================
    t["DPD_LIST"] = t["HISTORY"].apply(parse_history)

    all_dpd = flatten(t["DPD_LIST"])

    # windows
    last3  = all_dpd[:3]
    last6  = all_dpd[:6]
    last12 = all_dpd[:12]
    last24 = all_dpd[:24]
    last36 = all_dpd[:36]

    # =============================
    # FLAGS
    # =============================
    acct = t["ACCT-TYPE"].astype(str)
    sec  = t["SECURITY-STATUS"].astype(str)

    is_cc = acct.str.contains("CARD", case=False, na=False)
    is_non_cc = ~is_cc

    is_sec = sec.str.contains("SEC", case=False, na=False)
    is_unsec = sec.str.contains("UN", case=False, na=False)

    active = t["CLOSED-DT"].isna()

    # helper
    def dpd_mask(mask):
        return [x for i,row in t.iterrows() if mask[i] for x in row["DPD_LIST"]]

    cc_dpd = dpd_mask(is_cc)
    non_cc_dpd = dpd_mask(is_non_cc)
    sec_dpd = dpd_mask(is_sec)
    unsec_dpd = dpd_mask(is_unsec)
    active_dpd = dpd_mask(active)

    # =============================
    # VARIABLES
    # =============================
    r = {}
    r["pan"] = pan

    # core
    r["max_dpd_ever"] = max0(all_dpd)
    r["most_recent_dpd"] = all_dpd[0] if all_dpd else 0

    # time
    r["max_dpd_last_6_mo"] = max0(last6)
    r["max_dpd_inlast12months"] = max0(last12)
    r["max_dpd_inlast24months"] = max0(last24)
    r["max_dpd_inlast36months"] = max0(last36)

    # counts
    r["cnt_30plus_dpd_last6months"] = sum(x>=30 for x in last6)
    r["cnt_30plus_dpd_last24months"] = sum(x>=30 for x in last24)
    r["cnt_90plus_dpd_last24months"] = sum(x>=90 for x in last24)
    r["cnt_90plus_dpd_last36months"] = sum(x>=90 for x in last36)

    r["cnt_0dpd_last_6mo"] = sum(x==0 for x in last6)
    r["cnt_0dpd_last_12mo"] = sum(x==0 for x in last12)

    # active
    r["max_dpd_current_active_account"] = max0(active_dpd)

    # cc / non-cc
    r["max_dpd_cc_inlast6months"] = max0(cc_dpd[:6])
    r["max_dpd_non_cc_inlast6months"] = max0(non_cc_dpd[:6])

    # secured / unsecured
    r["max_dpd_secured_ever"] = max0(sec_dpd)
    r["max_dpd_unsecured_ever"] = max0(unsec_dpd)

    # write-off flag (FIXED)
    r["written_off_flag"] = int(t["WRITE-OFF-AMT"].sum() > 0)

    # percentage default
    total_inst = len(all_dpd)
    dpd_inst = sum(x>0 for x in all_dpd)

    r["percentage_installments_defaulted"] = (dpd_inst/total_inst*100) if total_inst else 0

    final.append(r)

# =============================
# OUTPUT
# =============================
final_df = pd.DataFrame(final)

save_path = r"C:\Users\Admin\Downloads\DPD_OUTPUT.xlsx"
final_df.to_excel(save_path, index=False)

print("✅ DONE")
print("Saved:", save_path)
print("Shape:", final_df.shape)
print("Columns:", final_df.columns.tolist())

✅ DONE
Saved: C:\Users\Admin\Downloads\DPD_OUTPUT.xlsx
Shape: (99, 20)
Columns: ['pan', 'max_dpd_ever', 'most_recent_dpd', 'max_dpd_last_6_mo', 'max_dpd_inlast12months', 'max_dpd_inlast24months', 'max_dpd_inlast36months', 'cnt_30plus_dpd_last6months', 'cnt_30plus_dpd_last24months', 'cnt_90plus_dpd_last24months', 'cnt_90plus_dpd_last36months', 'cnt_0dpd_last_6mo', 'cnt_0dpd_last_12mo', 'max_dpd_current_active_account', 'max_dpd_cc_inlast6months', 'max_dpd_non_cc_inlast6months', 'max_dpd_secured_ever', 'max_dpd_unsecured_ever', 'written_off_flag', 'percentage_installments_defaulted']


In [26]:
# =============================
# PREP EXTRA WINDOWS
# =============================
last3  = all_dpd[:3]
last6  = all_dpd[:6]
last12 = all_dpd[:12]
last18 = all_dpd[:18]
last24 = all_dpd[:24]
last36 = all_dpd[:36]

# masked dpd
def dpd_mask(mask):
    return [x for i,row in t.iterrows() if mask[i] for x in row["DPD_LIST"]]

cc_dpd = dpd_mask(is_cc)
non_cc_dpd = dpd_mask(is_non_cc)
sec_dpd = dpd_mask(is_sec)
unsec_dpd = dpd_mask(is_unsec)
active_dpd = dpd_mask(active)

active_cc_dpd = dpd_mask(active & is_cc)
active_non_cc_dpd = dpd_mask(active & is_non_cc)

# =============================
# VARIABLES
# =============================

r["most_recent_dpd_last12months"] = max(last12) if last12 else 0

# ACTIVE WINDOWS
r["max_dpd_active_account_last_3_mo"] = max(active_dpd[:3]) if active_dpd else 0
r["max_dpd_active_account_last_6_mo"] = max(active_dpd[:6]) if active_dpd else 0
r["max_dpd_active_account_last_12_mo"] = max(active_dpd[:12]) if active_dpd else 0
r["max_dpd_active_account_last_24_mo"] = max(active_dpd[:24]) if active_dpd else 0
r["max_dpd_active_account_last_36_mo"] = max(active_dpd[:36]) if active_dpd else 0

# CC
r["max_dpd_cc_last_3mo"] = max(cc_dpd[:3]) if cc_dpd else 0
r["max_dpd_cc_last_6mo"] = max(cc_dpd[:6]) if cc_dpd else 0
r["max_dpd_cc_last_12mo"] = max(cc_dpd[:12]) if cc_dpd else 0

# NON CC
r["max_dpd_non_cc_last_3mo"] = max(non_cc_dpd[:3]) if non_cc_dpd else 0
r["max_dpd_non_cc_last_6mo"] = max(non_cc_dpd[:6]) if non_cc_dpd else 0
r["max_dpd_non_cc_last_12mo"] = max(non_cc_dpd[:12]) if non_cc_dpd else 0

# PL (assume personal loan)
is_pl = t["ACCT-TYPE"].str.contains("PERSONAL", case=False, na=False)
pl_dpd = dpd_mask(is_pl)
r["max_dpd_pl_inlast12months"] = max(pl_dpd[:12]) if pl_dpd else 0

# LAST 3 MONTH PAYMENT DPD
r["last3months_pymt_max_dpd"] = max(last3) if last3 else 0
r["last3pymt_max_dpd"] = max(last3) if last3 else 0

# SECURED
r["max_dpd_secured_1_year"] = max(sec_dpd[:12]) if sec_dpd else 0
r["max_dpd_secured_inlast6months"] = max(sec_dpd[:6]) if sec_dpd else 0

# UNSECURED
r["max_dpd_unsecured_1_year"] = max(unsec_dpd[:12]) if unsec_dpd else 0
r["max_dpd_unsecured_inlast6months"] = max(unsec_dpd[:6]) if unsec_dpd else 0

# DATE PLACEHOLDERS (no true mapping available)
r["last3pymt_max_dpd_date"] = None
r["last3pymt_max_dpd_last12months"] = max(last12) if last12 else 0
r["last3pymt_max_dpd_last12months_date"] = None

# INSTALLMENT STATS
r["payments_reported_zero_dpd"] = sum(x == 0 for x in all_dpd)

r["cnt_dpd_installments"] = sum(x > 0 for x in all_dpd)
r["cnt_paid_dpd_installments"] = sum(x > 0 for x in all_dpd)
r["cnt_unpaid_dpd_installments"] = sum(x > 0 for x in all_dpd)

# CC COUNTS
r["cnt_cc_30plus_dpd_last6months"] = sum(x >= 30 for x in cc_dpd[:6])
r["cnt_cc_60plus_dpd_last6months"] = sum(x >= 60 for x in cc_dpd[:6])
r["cnt_cc_90plus_dpd_last6months"] = sum(x >= 90 for x in cc_dpd[:6])

r["cnt_cc_60plus_dpd_last12months"] = sum(x >= 60 for x in cc_dpd[:12])

# CC HIGH LIMIT FILTER
cc_high = t["CREDIT-LIMIT"] > 50000
cc_high_dpd = dpd_mask(is_cc & cc_high)
r["cnt_cc_50k_90plus_dpd_last6months"] = sum(x >= 90 for x in cc_high_dpd[:6])

r["cnt_cc_cl_30plus_dpd_last12months"] = sum(x >= 30 for x in cc_dpd[:12])

# NON CC LOGIC
r["cnt_non_cc_gl_tl_30plus_dpd_last12months"] = sum(x >= 30 for x in non_cc_dpd[:12])

# OWNERSHIP FILTER
own = t["OWNERSHIP-TYPE"].astype(str)
is_ind_joint = own.str.contains("IND|JOINT", case=False, na=False)

non_cc_ind_joint_dpd = dpd_mask(is_non_cc & is_ind_joint)

r["cnt_non_cc_individual_joint_30plus_dpd_last3months"] = sum(x >= 30 for x in non_cc_ind_joint_dpd[:3])
r["cnt_non_cc_individual_joint_15plus_dpd_last6months"] = sum(x >= 15 for x in non_cc_ind_joint_dpd[:6])
r["cnt_non_cc_individual_joint_30plus_dpd_last12months"] = sum(x >= 30 for x in non_cc_ind_joint_dpd[:12])
r["cnt_non_cc_individual_joint_30plus_dpd_last18months"] = sum(x >= 30 for x in non_cc_ind_joint_dpd[:18])
r["cnt_non_cc_individual_joint_30plus_dpd_last6months"] = sum(x >= 30 for x in non_cc_ind_joint_dpd[:6])

# NON CL (exclude consumer loan)
is_cl = t["ACCT-TYPE"].str.contains("CONSUMER", case=False, na=False)
non_cc_non_cl_dpd = dpd_mask(is_non_cc & ~is_cl)

r["cnt_non_cc_non_cl_30plus_dpd_last24months"] = sum(x >= 30 for x in non_cc_non_cl_dpd[:24])

# MAX AGAIN
r["max_dpd_cc_inlast12months"] = max(cc_dpd[:12]) if cc_dpd else 0
r["max_dpd_cc_inlast3months"] = max(cc_dpd[:3]) if cc_dpd else 0

r["max_dpd_non_cc_inlast12months"] = max(non_cc_dpd[:12]) if non_cc_dpd else 0
r["max_dpd_non_cc_inlast3months"] = max(non_cc_dpd[:3]) if non_cc_dpd else 0

# ACTIVE SPLIT
r["max_dpd_active_cc_last_3mo"] = max(active_cc_dpd[:3]) if active_cc_dpd else 0
r["max_dpd_active_cc_last_6mo"] = max(active_cc_dpd[:6]) if active_cc_dpd else 0
r["max_dpd_active_cc_last_12mo"] = max(active_cc_dpd[:12]) if active_cc_dpd else 0

r["max_dpd_active_non_cc_last_3mo"] = max(active_non_cc_dpd[:3]) if active_non_cc_dpd else 0
r["max_dpd_active_non_cc_last_6mo"] = max(active_non_cc_dpd[:6]) if active_non_cc_dpd else 0
r["max_dpd_active_non_cc_last_12mo"] = max(active_non_cc_dpd[:12]) if active_non_cc_dpd else 0

# ACTIVE LOANS IND/JOINT
active_ind_joint = active & is_ind_joint
active_ind_joint_dpd = dpd_mask(active_ind_joint)

r["cnt_active_loans_individual_joint_30plus_dpd_last12months"] = sum(x >= 30 for x in active_ind_joint_dpd[:12])

# NON GUARANTOR
is_guarantor = own.str.contains("GUARANTOR", case=False, na=False)
non_guarantor_dpd = dpd_mask(~is_guarantor)

r["max_dpd_inlast6months_non_guarantor"] = max(non_guarantor_dpd[:6]) if non_guarantor_dpd else 0

In [27]:
print(final_df.shape)

(99, 20)


In [28]:
rows.append(r)

In [29]:
# =============================
# MAIN LOOP
# =============================
rows = []

for raw in df["api_response"]:

    ...
    r = {}

    # -------------------------
    # FIRST VARIABLES
    # -------------------------
    r["max_dpd_ever"] = ...
    r["most_recent_dpd"] = ...

    # -------------------------
    # 🔥 ADD YOUR 51 VARIABLE BLOCK HERE
    # -------------------------
    # (paste the full block I gave)

    # -------------------------
    # SAVE ROW (LAST LINE ONLY)
    # -------------------------
    rows.append(r)

In [30]:
final_df = pd.DataFrame(rows)

print("Columns:", len(final_df.columns))
print(final_df.columns.tolist())

Columns: 2
['max_dpd_ever', 'most_recent_dpd']


In [31]:
print(len(r))

2


In [32]:
import pandas as pd
import json, ast

# =============================
# LOAD FILE
# =============================
path = r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv"
df = pd.read_csv(path, low_memory=False)

# =============================
# HELPERS
# =============================
def safe_json(x):
    try:
        return json.loads(x)
    except:
        try:
            return json.loads(x.replace('""','"'))
        except:
            try:
                return ast.literal_eval(x)
            except:
                return None

def num(x):
    try:
        return float(str(x).replace(",", "").strip())
    except:
        return 0.0

def parse_history(hist):
    dpd_list = []
    if not hist:
        return dpd_list

    for block in hist:
        vals = str(block.get("VALUES", ""))
        for p in vals.split("|"):
            if "/" in p:
                d = p.split("/")[0].strip()

                if d in ["STD","XXX","","NAN"]:
                    dpd_list.append(0)
                else:
                    try:
                        v = int(d)
                        if v > 365:
                            v = 0
                        dpd_list.append(v)
                    except:
                        dpd_list.append(0)
    return dpd_list

def flatten(lst):
    return [x for sub in lst for x in sub]

# =============================
# MAIN LOOP
# =============================
rows = []

for raw in df["api_response"]:

    data = safe_json(str(raw))
    if not isinstance(data, dict):
        continue

    try:
        block = data["data"]
        pan = block.get("pan")

        tradelines = (
            block.get("credit_report", {})
                 .get("CIR-REPORT-FILE", {})
                 .get("REPORT-DATA", {})
                 .get("STANDARD-DATA", {})
                 .get("TRADELINES", [])
        )
    except:
        continue

    if not tradelines:
        continue

    t = pd.DataFrame(tradelines)
    if t.empty:
        continue

    # =============================
    # CLEAN NUMERIC
    # =============================
    for col in ["CREDIT-LIMIT","WRITE-OFF-AMT"]:
        if col in t.columns:
            t[col] = t[col].apply(num)
        else:
            t[col] = 0

    # =============================
    # DPD
    # =============================
    t["DPD_LIST"] = t["HISTORY"].apply(parse_history)
    all_dpd = flatten(t["DPD_LIST"])

    if len(all_dpd) == 0:
        continue

    # =============================
    # FLAGS
    # =============================
    acct = t["ACCT-TYPE"].astype(str)
    sec  = t["SECURITY-STATUS"].astype(str)
    own  = t["OWNERSHIP-TYPE"].astype(str)

    is_cc = acct.str.contains("CARD", case=False, na=False)
    is_non_cc = ~is_cc
    is_sec = sec.str.contains("SEC", case=False, na=False)
    is_unsec = sec.str.contains("UN", case=False, na=False)
    active = t["CLOSED-DT"].isna()
    is_ind_joint = own.str.contains("IND|JOINT", case=False, na=False)

    def dpd_mask(mask):
        return [x for i,row in t.iterrows() if mask[i] for x in row["DPD_LIST"]]

    cc_dpd = dpd_mask(is_cc)
    non_cc_dpd = dpd_mask(is_non_cc)
    sec_dpd = dpd_mask(is_sec)
    unsec_dpd = dpd_mask(is_unsec)
    active_dpd = dpd_mask(active)
    active_cc_dpd = dpd_mask(active & is_cc)
    active_non_cc_dpd = dpd_mask(active & is_non_cc)

    # =============================
    # WINDOWS
    # =============================
    last3  = all_dpd[:3]
    last6  = all_dpd[:6]
    last12 = all_dpd[:12]
    last18 = all_dpd[:18]
    last24 = all_dpd[:24]
    last36 = all_dpd[:36]

    # =============================
    # BUILD RESULT
    # =============================
    r = {}
    r["pan"] = pan

    # BASIC
    r["max_dpd_ever"] = max(all_dpd)
    r["most_recent_dpd"] = all_dpd[0]
    r["most_recent_dpd_last12months"] = max(last12)

    # ACTIVE
    r["max_dpd_current_active_account"] = max(active_dpd) if active_dpd else 0
    r["max_dpd_active_account_last_3_mo"] = max(active_dpd[:3]) if active_dpd else 0
    r["max_dpd_active_account_last_6_mo"] = max(active_dpd[:6]) if active_dpd else 0

    # CC / NON CC
    r["max_dpd_cc_last_3mo"] = max(cc_dpd[:3]) if cc_dpd else 0
    r["max_dpd_non_cc_last_3mo"] = max(non_cc_dpd[:3]) if non_cc_dpd else 0

    # COUNTS
    r["cnt_30plus_dpd_last6months"] = sum(x>=30 for x in last6)
    r["cnt_90plus_dpd_last24months"] = sum(x>=90 for x in last24)

    # INSTALLMENTS
    r["cnt_dpd_installments"] = sum(x>0 for x in all_dpd)
    r["payments_reported_zero_dpd"] = sum(x==0 for x in all_dpd)

    # SECURED / UNSECURED
    r["max_dpd_secured_inlast6months"] = max(sec_dpd[:6]) if sec_dpd else 0
    r["max_dpd_unsecured_inlast6months"] = max(unsec_dpd[:6]) if unsec_dpd else 0

    # WRITE OFF
    r["written_off_flag"] = int(t["WRITE-OFF-AMT"].sum() > 0)

    # DEBUG CHECK
    print("PAN:", pan, "Vars:", len(r))

    rows.append(r)

# =============================
# FINAL OUTPUT
# =============================
final_df = pd.DataFrame(rows)

save_path = r"C:\Users\Admin\Downloads\FINAL_DPD_OUTPUT.xlsx"
final_df.to_excel(save_path, index=False)

print("✅ DONE")
print("Shape:", final_df.shape)
print("Columns:", len(final_df.columns))

PAN: KYEPS8060N Vars: 16
PAN: FEWPA9169H Vars: 16
PAN: ATOPB8483D Vars: 16
PAN: DSTPA0117A Vars: 16
PAN: BTZPG5521K Vars: 16
PAN: GHRPA4842N Vars: 16
PAN: AHHPH6651E Vars: 16
PAN: FEFPS7171C Vars: 16
PAN: AZAPD2389L Vars: 16
PAN: APVPP7852F Vars: 16
PAN: KXCPK5577D Vars: 16
PAN: MQLPK9471L Vars: 16
PAN: KSEPS7848A Vars: 16
PAN: BGQPN6469R Vars: 16
PAN: NARPS8252N Vars: 16
PAN: IMWPK2862N Vars: 16
PAN: SMCPS8940F Vars: 16
PAN: CTPPM6154F Vars: 16
PAN: CQHPP2279M Vars: 16
PAN: CMSPJ3453L Vars: 16
PAN: DVGPK5286N Vars: 16
PAN: AEVPI7465J Vars: 16
PAN: BRGPK4510H Vars: 16
PAN: AXPPA3224N Vars: 16
PAN: GGWPP0316L Vars: 16
PAN: DFIPA5637H Vars: 16
PAN: BWJPH2515K Vars: 16
PAN: GICPA9911A Vars: 16
PAN: AJHPL2037J Vars: 16
PAN: QEGPS2694A Vars: 16
PAN: BIDPR3882B Vars: 16
PAN: AXDPG5559N Vars: 16
PAN: BMKPG9959F Vars: 16
PAN: GMJPB2320H Vars: 16
PAN: NIFPK6835R Vars: 16
PAN: EABPA2302E Vars: 16
PAN: FKIPD5971G Vars: 16
PAN: AJOPT1012A Vars: 16
PAN: FVBPM0840M Vars: 16
PAN: CBMPA6175K Vars: 16


In [33]:
import pandas as pd
import json, ast

# =============================
# LOAD
# =============================
path = r"C:\Users\Admin\Downloads\kunal_bureau_variables_task.csv"
df = pd.read_csv(path, low_memory=False)

# =============================
# HELPERS
# =============================
def safe_json(x):
    try:
        return json.loads(x)
    except:
        try:
            return json.loads(x.replace('""','"'))
        except:
            try:
                return ast.literal_eval(x)
            except:
                return None

def num(x):
    try:
        return float(str(x).replace(",", "").strip())
    except:
        return 0.0

def parse_history(hist):
    dpd_list = []
    if not hist:
        return dpd_list

    for block in hist:
        vals = str(block.get("VALUES",""))
        for p in vals.split("|"):
            if "/" in p:
                d = p.split("/")[0].strip()
                if d in ["STD","XXX","","NAN"]:
                    dpd_list.append(0)
                else:
                    try:
                        v = int(d)
                        if v > 365:
                            v = 0
                        dpd_list.append(v)
                    except:
                        dpd_list.append(0)
    return dpd_list

def dpd_mask(t, mask):
    return [x for i,row in t.iterrows() if mask[i] for x in row["DPD_LIST"]]

# =============================
# MAIN LOOP
# =============================
rows = []

for raw in df["api_response"]:

    data = safe_json(str(raw))
    if not isinstance(data, dict):
        continue

    try:
        block = data["data"]
        pan = block.get("pan")

        tradelines = (
            block.get("credit_report", {})
                 .get("CIR-REPORT-FILE", {})
                 .get("REPORT-DATA", {})
                 .get("STANDARD-DATA", {})
                 .get("TRADELINES", [])
        )
    except:
        continue

    if not tradelines:
        continue

    t = pd.DataFrame(tradelines)
    if t.empty:
        continue

    # CLEAN
    if "CREDIT-LIMIT" in t.columns:
        t["CREDIT-LIMIT"] = t["CREDIT-LIMIT"].apply(num)
    else:
        t["CREDIT-LIMIT"] = 0

    # DPD
    t["DPD_LIST"] = t["HISTORY"].apply(parse_history)
    all_dpd = [x for sub in t["DPD_LIST"] for x in sub]

    if not all_dpd:
        continue

    # FLAGS
    acct = t["ACCT-TYPE"].astype(str)
    sec  = t["SECURITY-STATUS"].astype(str)
    own  = t["OWNERSHIP-TYPE"].astype(str)

    is_cc = acct.str.contains("CARD", case=False, na=False)
    is_non_cc = ~is_cc
    is_sec = sec.str.contains("SEC", case=False, na=False)
    is_unsec = sec.str.contains("UN", case=False, na=False)
    active = t["CLOSED-DT"].isna()
    is_ind_joint = own.str.contains("IND|JOINT", case=False, na=False)
    is_guarantor = own.str.contains("GUARANTOR", case=False, na=False)
    is_pl = acct.str.contains("PERSONAL", case=False, na=False)
    is_cl = acct.str.contains("CONSUMER", case=False, na=False)

    # MASKED DPD
    cc_dpd = dpd_mask(t, is_cc)
    non_cc_dpd = dpd_mask(t, is_non_cc)
    sec_dpd = dpd_mask(t, is_sec)
    unsec_dpd = dpd_mask(t, is_unsec)
    active_dpd = dpd_mask(t, active)
    active_cc_dpd = dpd_mask(t, active & is_cc)
    active_non_cc_dpd = dpd_mask(t, active & is_non_cc)
    pl_dpd = dpd_mask(t, is_pl)
    non_guar_dpd = dpd_mask(t, ~is_guarantor)

    # WINDOWS
    last3  = all_dpd[:3]
    last6  = all_dpd[:6]
    last12 = all_dpd[:12]
    last18 = all_dpd[:18]
    last24 = all_dpd[:24]
    last36 = all_dpd[:36]

    # RESULT
    r = {"pan": pan}

    r["most_recent_dpd_last12months"] = max(last12)
    r["max_dpd_active_account_last_3_mo"] = max(active_dpd[:3]) if active_dpd else 0
    r["max_dpd_active_account_last_6_mo"] = max(active_dpd[:6]) if active_dpd else 0
    r["max_dpd_active_account_last_12_mo"] = max(active_dpd[:12]) if active_dpd else 0
    r["max_dpd_active_account_last_24_mo"] = max(active_dpd[:24]) if active_dpd else 0
    r["max_dpd_active_account_last_36_mo"] = max(active_dpd[:36]) if active_dpd else 0

    r["max_dpd_cc_last_3mo"] = max(cc_dpd[:3]) if cc_dpd else 0
    r["max_dpd_cc_last_6mo"] = max(cc_dpd[:6]) if cc_dpd else 0
    r["max_dpd_cc_last_12mo"] = max(cc_dpd[:12]) if cc_dpd else 0

    r["max_dpd_non_cc_last_3mo"] = max(non_cc_dpd[:3]) if non_cc_dpd else 0
    r["max_dpd_non_cc_last_6mo"] = max(non_cc_dpd[:6]) if non_cc_dpd else 0
    r["max_dpd_non_cc_last_12mo"] = max(non_cc_dpd[:12]) if non_cc_dpd else 0

    r["max_dpd_pl_inlast12months"] = max(pl_dpd[:12]) if pl_dpd else 0

    r["last3months_pymt_max_dpd"] = max(last3)
    r["last3pymt_max_dpd"] = max(last3)

    r["max_dpd_secured_1_year"] = max(sec_dpd[:12]) if sec_dpd else 0
    r["max_dpd_secured_inlast6months"] = max(sec_dpd[:6]) if sec_dpd else 0

    r["max_dpd_unsecured_1_year"] = max(unsec_dpd[:12]) if unsec_dpd else 0
    r["max_dpd_unsecured_inlast6months"] = max(unsec_dpd[:6]) if unsec_dpd else 0

    r["last3pymt_max_dpd_last12months"] = max(last12)

    r["payments_reported_zero_dpd"] = sum(x == 0 for x in all_dpd)
    r["cnt_dpd_installments"] = sum(x > 0 for x in all_dpd)
    r["cnt_paid_dpd_installments"] = sum(x > 0 for x in all_dpd)
    r["cnt_unpaid_dpd_installments"] = sum(x > 0 for x in all_dpd)

    r["cnt_cc_30plus_dpd_last6months"] = sum(x>=30 for x in cc_dpd[:6])
    r["cnt_cc_60plus_dpd_last6months"] = sum(x>=60 for x in cc_dpd[:6])
    r["cnt_cc_90plus_dpd_last6months"] = sum(x>=90 for x in cc_dpd[:6])

    r["cnt_cc_60plus_dpd_last12months"] = sum(x>=60 for x in cc_dpd[:12])

    cc_high = t["CREDIT-LIMIT"] > 50000
    cc_high_dpd = dpd_mask(t, is_cc & cc_high)
    r["cnt_cc_50k_90plus_dpd_last6months"] = sum(x>=90 for x in cc_high_dpd[:6])

    r["cnt_cc_cl_30plus_dpd_last12months"] = sum(x>=30 for x in cc_dpd[:12])

    r["cnt_non_cc_gl_tl_30plus_dpd_last12months"] = sum(x>=30 for x in non_cc_dpd[:12])

    non_cc_ind = dpd_mask(t, is_non_cc & is_ind_joint)
    r["cnt_non_cc_individual_joint_30plus_dpd_last3months"] = sum(x>=30 for x in non_cc_ind[:3])
    r["cnt_non_cc_individual_joint_15plus_dpd_last6months"] = sum(x>=15 for x in non_cc_ind[:6])
    r["cnt_non_cc_individual_joint_30plus_dpd_last12months"] = sum(x>=30 for x in non_cc_ind[:12])
    r["cnt_non_cc_individual_joint_30plus_dpd_last18months"] = sum(x>=30 for x in non_cc_ind[:18])
    r["cnt_non_cc_individual_joint_30plus_dpd_last6months"] = sum(x>=30 for x in non_cc_ind[:6])

    non_cc_non_cl = dpd_mask(t, is_non_cc & ~is_cl)
    r["cnt_non_cc_non_cl_30plus_dpd_last24months"] = sum(x>=30 for x in non_cc_non_cl[:24])

    r["max_dpd_cc_inlast12months"] = max(cc_dpd[:12]) if cc_dpd else 0
    r["max_dpd_cc_inlast3months"] = max(cc_dpd[:3]) if cc_dpd else 0

    r["max_dpd_non_cc_inlast12months"] = max(non_cc_dpd[:12]) if non_cc_dpd else 0
    r["max_dpd_non_cc_inlast3months"] = max(non_cc_dpd[:3]) if non_cc_dpd else 0

    r["max_dpd_active_cc_last_3mo"] = max(active_cc_dpd[:3]) if active_cc_dpd else 0
    r["max_dpd_active_cc_last_6mo"] = max(active_cc_dpd[:6]) if active_cc_dpd else 0
    r["max_dpd_active_cc_last_12mo"] = max(active_cc_dpd[:12]) if active_cc_dpd else 0

    r["max_dpd_active_non_cc_last_3mo"] = max(active_non_cc_dpd[:3]) if active_non_cc_dpd else 0
    r["max_dpd_active_non_cc_last_6mo"] = max(active_non_cc_dpd[:6]) if active_non_cc_dpd else 0
    r["max_dpd_active_non_cc_last_12mo"] = max(active_non_cc_dpd[:12]) if active_non_cc_dpd else 0

    active_ind = dpd_mask(t, active & is_ind_joint)
    r["cnt_active_loans_individual_joint_30plus_dpd_last12months"] = sum(x>=30 for x in active_ind[:12])

    r["max_dpd_inlast6months_non_guarantor"] = max(non_guar_dpd[:6]) if non_guar_dpd else 0

    rows.append(r)

# =============================
# SAVE FILE
# =============================
final_df = pd.DataFrame(rows)

save_path = r"C:\Users\Admin\Downloads\FINAL_51_VARIABLES.xlsx"
final_df.to_excel(save_path, index=False)

print("✅ DONE")
print("Shape:", final_df.shape)

✅ DONE
Shape: (99, 50)
